In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [2]:
df1 = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
df2 = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
df = df2.merge(df1,on=['TransactionID'],how='left')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), object(31)
memory usage: 1.9+ GB


In [3]:
for col in df.columns:
    print(f"Column '{col}' has {df[col].isna().sum()} missing values.")

Column 'TransactionID' has 0 missing values.
Column 'isFraud' has 0 missing values.
Column 'TransactionDT' has 0 missing values.
Column 'TransactionAmt' has 0 missing values.
Column 'ProductCD' has 0 missing values.
Column 'card1' has 0 missing values.
Column 'card2' has 8933 missing values.
Column 'card3' has 1565 missing values.
Column 'card4' has 1577 missing values.
Column 'card5' has 4259 missing values.
Column 'card6' has 1571 missing values.
Column 'addr1' has 65706 missing values.
Column 'addr2' has 65706 missing values.
Column 'dist1' has 352271 missing values.
Column 'dist2' has 552913 missing values.
Column 'P_emaildomain' has 94456 missing values.
Column 'R_emaildomain' has 453249 missing values.
Column 'C1' has 0 missing values.
Column 'C2' has 0 missing values.
Column 'C3' has 0 missing values.
Column 'C4' has 0 missing values.
Column 'C5' has 0 missing values.
Column 'C6' has 0 missing values.
Column 'C7' has 0 missing values.
Column 'C8' has 0 missing values.
Column 'C9

In [4]:
categorical_cols = [col for col in df.columns if df[col].dtype == "object"]
categorical_cols

['ProductCD',
 'card4',
 'card6',
 'P_emaildomain',
 'R_emaildomain',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'id_12',
 'id_15',
 'id_16',
 'id_23',
 'id_27',
 'id_28',
 'id_29',
 'id_30',
 'id_31',
 'id_33',
 'id_34',
 'id_35',
 'id_36',
 'id_37',
 'id_38',
 'DeviceType',
 'DeviceInfo']

# Cleaning

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class DealWithNans(BaseEstimator, TransformerMixin):
    def __init__(self, drop_threshold=0.8):
        self.drop_threshold = drop_threshold
        self.fill_values_ = {}
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        self.cols_to_drop_ = [
            col for col in X.columns if X[col].isna().mean() >= self.drop_threshold
        ]

        X_filtered = X.drop(columns=self.cols_to_drop_)

        for col in X_filtered.columns:
            if X_filtered[col].dtype == "object" or str(X_filtered[col].dtype).startswith("category"):
                # Categorical: use mode
                mode = X_filtered[col].mode()
                self.fill_values_[col] = mode.iloc[0] if not mode.empty else np.nan
            else:
                # Numeric: use median
                self.fill_values_[col] = X_filtered[col].median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.drop(columns=self.cols_to_drop_, errors="ignore")
        return X.fillna(self.fill_values_)


In [65]:
from sklearn.model_selection import GroupKFold

X = df.drop(columns=['isFraud'])
y = df['isFraud']

groups = X['card2'].astype(str) + "-" + X['addr2'].astype(str) + "-" + X['C1'].astype(str)

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)


In [13]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((472432, 433), (472432,), (118108, 433), (118108,))

In [71]:
cleaner = DealWithNans(drop_threshold = 0.90)
cleaner.fit(X_train, y_train)
X_no_nans = cleaner.transform(X_train)
X_no_nans.shape

(472432, 421)

In [20]:
!pip install mlflow dagshub
import mlflow
import dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 55.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 80.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.3/692.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
 

In [21]:
dagshub.init(repo_owner='mr-master-afk', repo_name='ML-Fraud-detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4048f71b-f839-473f-bf29-3d31f186cb4a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8b4c150a30cd6380f995b32e39aca58d6b802831656f9188ad7d066ff1b5a6f8




Output()

Accessing as mr-master-afk

Initialized MLflow to track repo "mr-master-afk/ML-Fraud-detection"

Repository mr-master-afk/ML-Fraud-detection initialized!

In [22]:
experiment_name = "LogisticRegression"
run_name = "logistic_regression_cleaning"
# Set the experiment name
mlflow.set_experiment(experiment_name)
removed_col_cnt = X_train.shape[1] - X_no_nans.shape[1]
with mlflow.start_run(run_name=run_name):
    mlflow.log_param("drop_threshold", 0.85)
    mlflow.log_param("dropped_columns_cnt", removed_col_cnt)
    mlflow.sklearn.log_model(
        cleaner,
        artifact_path="cleaner_model",
        registered_model_name="Cleaner",
    )
mlflow.end_run()

2025/04/29 13:01:18 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression' does not exist. Creating a new experiment.
2025/04/29 13:01:20 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/04/29 13:01:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Cleaner'.
2025/04/29 13:01:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Cleaner, version 1
Created version '1' of model 'Cleaner'.


🏃 View run logistic_regression_cleaning at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0/runs/1e11d80f85c2432cac801c60bea2c891
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0


In [19]:
X_no_nans.shape

(472432, 359)

# Feature Engineering

In [60]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CustomFeatureEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for feature in X.select_dtypes(include=['object', 'category']).columns:
            encoder = LabelEncoder()
            encoder.fit(X[feature])
            self.label_encoders[feature] = encoder
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for feature, encoder in self.label_encoders.items():
            unseen_labels = ~X_transformed[feature].isin(encoder.classes_)
            if unseen_labels.any():
                X_transformed.loc[unseen_labels, feature] = encoder.classes_[0]
            X_transformed[feature] = encoder.transform(X_transformed[feature])

        return X_transformed


In [61]:
scaler_and_encoder = Pipeline([
    ('encoder', CustomFeatureEncoder()),
    ('scaler', StandardScaler())
])
scaler_and_encoder.fit(X_no_nans, y_train)
X_enc_sc = scaler_and_encoder.transform(X_no_nans)
with mlflow.start_run(run_name="Encoder_and_Scaler") as run:
    mlflow.sklearn.log_model(
        scaler_and_encoder,
        artifact_path="encoder_and_scaler_model",
        registered_model_name="Encoder_and_Scaler",
    )
mlflow.end_run()

Pipeline(steps=[('encoder', CustomFeatureEncoder()),
                ('scaler', StandardScaler())])

# Feature Selection

In [39]:
class CorrelationFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cnt):
        self.cnt = cnt
        self.selected_features = []

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = pd.Series(y)

        corrs = X.apply(lambda col: np.abs(np.corrcoef(col, y)[0, 1]))
        
        self.selected_features = corrs.nlargest(self.cnt).index.tolist()
        return self

    def transform(self, X):
        return pd.DataFrame(X)[self.selected_features]

In [40]:
corr_feature_selector = CorrelationFeatureSelector(120)
corr_feature_selector.fit(X_enc_sc, y_train)
X_corr_selected = corr_feature_selector.transform(X_enc_sc)
if mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(run_name="Feature_Selection_Correlation") as run:
    mlflow.sklearn.log_model(
        corr_feature_selector,
        artifact_path="feature_selector_corr",
        registered_model_name="Feature_Selection_Corr",
    )
mlflow.log_param("count_of_features", corr_feature_selector.cnt)
mlflow.end_run()


2025/04/29 14:09:44 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/04/29 14:09:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Feature_Selection_Corr' already exists. Creating a new version of this model...
2025/04/29 14:09:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Feature_Selection_Corr, version 4
Created version '4' of model 'Feature_Selection_Corr'.


🏃 View run Feature_Selection_Correlation at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0/runs/e225a8f6ca82443f8d617717d70e22cd
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0


120

# Training

In [42]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, solver='liblinear')

model.fit(X_corr_selected, y_train)

LogisticRegression(max_iter=1000, solver='liblinear')

In [62]:
X_test_cleaned = cleaner.transform(X_test)
X_test_enc_scaled = scaler_and_encoder.transform(X_test_cleaned)
X_test_corr_selected = corr_feature_selector.transform(X_test_enc_scaled)

In [68]:
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_corr_selected)[:, 1]
roc_score = roc_auc_score(y_test, y_prob)
with mlflow.start_run(run_name="Logistic_Regression_Model") as run:
    mlflow.sklearn.log_model(
        model,
        artifact_path="logistic_regression_model",
        registered_model_name="LogisticRegressionModel"
    )
    mlflow.log_param("max_iter", model.max_iter)
    mlflow.log_metric("roc_auc_score", roc_score)
    
mlflow.end_run()

2025/04/29 15:34:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'LogisticRegressionModel' already exists. Creating a new version of this model...
2025/04/29 15:34:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LogisticRegressionModel, version 4
Created version '4' of model 'LogisticRegressionModel'.


🏃 View run Logistic_Regression_Model at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0/runs/62eac5210bb8442b8c9bb9de29dcba80
🧪 View experiment at: https://dagshub.com/mr-master-afk/ML-Fraud-detection.mlflow/#/experiments/0
